# Disciplina de Inteligência Artificial
## Trabalho de Aprendizado Supervisionado — Etapa 1: Análise Exploratória dos Dados (EDA) e Investigação de Problemas

### 1. Descrição do Problema
* **Objetivo Geral:** Desenvolver um pipeline completo de Machine Learning e Redes Neurais Multicamadas (MLP) aplicado a um problema real, contemplando análise exploratória dos dados, pré-processamento, engenharia de atributos, classificação, regressão, comparação com métodos tradicionais, otimização de hiperparâmetros e análise de generalização.
* **Problema:** Predição de atropelamentos e identificação de *hotspots* de fauna silvestre (dados do Projeto Rodofauna / IBRAM-DF).
* **Atributos Esperados na Base de Dados:**
  * `id`, `campanha`, `ano`, `periodo`, `mes`, `data_dt`, `dia_da_sem`
  * `silvestre_` (tipo de fauna), `classe`, `ordem`, `familia`, `especie_nm`, `nome_popul`
  * `uc` (Unidade de Conservação), `x` (coordenada UTM Leste), `y` (coordenada UTM Norte), `posicao`, `hora`, `sexo`

### 2. Importação de Bibliotecas e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (12, 6)

# Carregamento do dataset (ajuste o nome/caminho do ficheiro caso utilize CSV ou Excel)
ARQUIVO_DADOS = 'rodofauna.csv' # ou 'rodofauna.xlsx'

try:
    if ARQUIVO_DADOS.endswith('.csv'):
        try:
            df = pd.read_csv(ARQUIVO_DADOS, sep=None, engine='python', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(ARQUIVO_DADOS, sep=None, engine='python', encoding='latin1')
    else:
        df = pd.read_excel(ARQUIVO_DADOS)
    print(f"Base de dados '{ARQUIVO_DADOS}' carregada com sucesso!")
except FileNotFoundError:
    print(f"Aviso: Ficheiro '{ARQUIVO_DADOS}' não encontrado no diretório atual.")
    print("A gerar um DataFrame de exemplo com a estrutura exata do problema para demonstração do código...")
    np.random.seed(42)
    n_samples = 500
    df = pd.DataFrame({
        'id': range(1, n_samples + 1),
        'campanha': np.random.randint(1, 15, n_samples),
        'ano': np.random.choice([2010, 2011, 2012, 2013, 2014, 2015], n_samples, p=[0.15, 0.2, 0.2, 0.2, 0.15, 0.1]),
        'periodo': np.random.choice([1, 2], n_samples),
        'mes': np.random.choice(['janeiro', 'fevereiro', 'marco', 'abril', 'maio', 'junho', 
                                 'julho', 'agosto', 'setembro', 'outubro', 'novembro', 'dezembro'], n_samples),
        'data_dt': pd.date_range('2010-01-01', periods=n_samples, freq='D').strftime('%d/%m/%y'),
        'dia_da_sem': np.random.choice(['seg', 'ter', 'qua', 'qui', 'sex', 'sab', 'dom'], n_samples),
        'silvestre_': np.random.choice(['silvestre', 'domestico', 'exotico', 'nao_identificado'], n_samples, p=[0.80, 0.12, 0.05, 0.03]),
        'classe': np.random.choice(['birds', 'mammalia', 'reptilia', 'amphibia', 'nao_identificado'], n_samples, p=[0.42, 0.35, 0.13, 0.04, 0.06]),
        'ordem': np.random.choice(['apodiformes', 'carnivora', 'rodentia', 'squamata', 'passeriformes', 'anura', 'nao_identificado'], n_samples),
        'familia': np.random.choice(['apodidae', 'canidae', 'caviidae', 'colubridae', 'tyrannidae', 'bufonidae', 'nao_identificado'], n_samples),
        'especie_nm': np.random.choice(['streptoprocne_zonaris', 'cerdocyon_thous', 'cavia_aperea', 'boa_constrictor', 'pitangus_sulphuratus', 'nao_identificado'], n_samples, p=[0.1, 0.25, 0.15, 0.05, 0.05, 0.40]),
        'nome_popul': np.random.choice(['andorinhao_de_coleira', 'cachorro_do_mato', 'prea', 'jiboia', 'bem_te_vi', 'nao_identificado'], n_samples, p=[0.1, 0.25, 0.15, 0.05, 0.05, 0.40]),
        'uc': np.random.choice(['esecae', 'parna_brasilia', 'apa_cafuringa', 'rebio_contagem', 'nao_identificado'], n_samples, p=[0.35, 0.30, 0.15, 0.10, 0.10]),
        'x': np.random.normal(219467, 4500, n_samples).astype(int),
        'y': np.random.normal(8273815, 6500, n_samples).astype(int),
        'posicao': np.random.choice(['pista', 'acostamento', 'canteiro_central', 'nao_identificado'], n_samples, p=[0.35, 0.25, 0.05, 0.35]),
        'hora': np.random.choice(['manha', 'tarde', 'noite', 'madrugada', 'nao_identificado'], n_samples, p=[0.15, 0.10, 0.20, 0.15, 0.40]),
        'sexo': np.random.choice(['macho', 'femea', 'nao_identificado'], n_samples, p=[0.18, 0.17, 0.65])
    })
    # Inserir alguns outliers simulados nas coordenadas e valores ausentes (NaN)
    df.loc[np.random.choice(df.index, 5), 'x'] = 350000
    df.loc[np.random.choice(df.index, 12), 'uc'] = np.nan
    df.loc[np.random.choice(df.index, 8), 'posicao'] = np.nan

# Padronizar nomes das colunas (remover espaços extras e converter para minúsculas)
df.columns = df.columns.str.strip().str.lower()
display(df.head(10))

---
### 3. Quantidade de Registos, Quantidade de Atributos e Tipos dos Atributos

In [ ]:
num_registos, num_atributos = df.shape
print(f"Quantidade de registos (linhas): {num_registos}")
print(f"Quantidade de atributos (colunas): {num_atributos}\n")

# Resumo detalhado dos tipos de dados e valores únicos
resumo_atributos = pd.DataFrame({
    'Tipo de Dado (Pandas)': df.dtypes,
    'Classificação Estatística': ['Numérico' if pd.api.types.is_numeric_dtype(df[col]) else 'Categórico / Texto' for col in df.columns],
    'Valores Não-Nulos': df.notnull().sum(),
    'Cardinalidade (Valores Únicos)': df.nunique(),
    'Exemplo de Valor': [df[col].dropna().iloc[0] if not df[col].dropna().empty else None for col in df.columns]
})

display(resumo_atributos)

---
### 4. Quantidade de Valores Ausentes (`NaN`) e Valores `nao_identificado`
Além dos valores nulos padrão (`NaN` / `None`), a base de dados codifica informações desconhecidas com a string `'nao_identificado'` (ou variações). Abaixo avaliamos ambos os cenários.

In [ ]:
# Contagem de valores ausentes (NaN)
nulos_nan = df.isnull().sum()
perc_nan = (nulos_nan / len(df)) * 100

# Contagem de 'nao_identificado' (case-insensitive e tratando com/sem acento)
padroes_ni = ['nao_identificado', 'não_identificado', 'nao identificado', 'não identificado', 'ni', 'indeterminado']
contagem_ni = {}

for col in df.columns:
    if df[col].dtype == 'object':
        contagem_ni[col] = df[col].astype(str).str.strip().str.lower().isin(padroes_ni).sum()
    else:
        contagem_ni[col] = 0

serie_ni = pd.Series(contagem_ni)
perc_ni = (serie_ni / len(df)) * 100

df_ausentes = pd.DataFrame({
    'Ausentes (NaN)': nulos_nan,
    '% Ausentes (NaN)': perc_nan.round(2),
    'Valores "nao_identificado"': serie_ni,
    '% "nao_identificado"': perc_ni.round(2),
    'Total Desconhecido (NaN + NI)': nulos_nan + serie_ni,
    '% Total Desconhecido': (perc_nan + perc_ni).round(2)
}).sort_values(by='% Total Desconhecido', ascending=False)

display(df_ausentes)

# Visualização gráfica dos atributos com valores ausentes ou nao_identificado
df_plot_ausentes = df_ausentes[df_ausentes['% Total Desconhecido'] > 0]

if not df_plot_ausentes.empty:
    ax = df_plot_ausentes[['% Ausentes (NaN)', '% "nao_identificado"']].plot(
        kind='barh', stacked=True, figsize=(10, 6), color=['#e74c3c', '#f39c12']
    )
    plt.title('Percentagem de Valores Ausentes (NaN) e "nao_identificado" por Atributo', fontsize=14)
    plt.xlabel('Percentagem (%)')
    plt.ylabel('Atributo')
    plt.axvline(30, color='red', linestyle='--', alpha=0.6, label='Limiar Crítico (30%)')
    plt.legend(loc='lower right')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

---
### 5. Variável Alvo de Cada Experimento e Distribuição das Classes

No contexto do problema de **Predição de Atropelamentos e Identificação de Hotspots de Fauna Silvestre**, os experimentos de Aprendizado Supervisionado dividem-se tipicamente em:

1. **Experimento de Classificação:**
   * **Variável Alvo Principal Sugerida:** `classe` (ex.: *birds*, *mammalia*, *reptilia*, *amphibia*) para prever o grupo taxonómico do animal atropelado dadas as condições espaço-temporais, ou `uc` / `silvestre_` / severidade do hotspot.
2. **Experimento de Regressão:**
   * **Variável Alvo Principal Sugerida:** Contagem/Frequência agregada de atropelamentos por trecho espacial (grid de coordenadas `x`, `y`), por campanha ou janela temporal (`ano`/`mes`/`periodo`), permitindo estimar a intensidade de atropelamentos e identificar *hotspots* críticos.

In [ ]:
# Análise da distribuição das potenciais variáveis alvo de Classificação
alvos_classificacao = [col for col in ['classe', 'silvestre_', 'uc', 'periodo'] if col in df.columns]

fig, axes = plt.subplots(1, len(alvos_classificacao), figsize=(6 * len(alvos_classificacao), 5))
if len(alvos_classificacao) == 1:
    axes = [axes]

for ax, col in zip(axes, alvos_classificacao):
    contagem = df[col].fillna('NaN').value_counts()
    perc = (contagem / len(df)) * 100
    
    sns.barplot(x=contagem.values, y=contagem.index.astype(str), ax=ax, palette='viridis')
    ax.set_title(f'Distribuição de "{col}"', fontsize=12, fontweight='bold')
    ax.set_xlabel('Quantidade de Registos')
    ax.set_ylabel(col)
    
    for i, (v, p) in enumerate(zip(contagem.values, perc.values)):
        ax.text(v + max(contagem.values)*0.01, i, f'{v} ({p:.1f}%)', va='center', fontsize=9)
    ax.set_xlim(0, max(contagem.values) * 1.22)

plt.tight_layout()
plt.show()

# Criação e visualização da variável alvo para o Experimento de Regressão (Contagem por Campanha/Mês/UC ou Grid Espacial)
if {'ano', 'mes', 'uc'}.issubset(df.columns):
    df_regressao_alvo = df.groupby(['ano', 'mes', 'uc']).size().reset_index(name='qtd_atropelamentos')
    plt.figure(figsize=(10, 4))
    sns.histplot(df_regressao_alvo['qtd_atropelamentos'], kde=True, bins=20, color='teal')
    plt.title('Distribuição da Variável Alvo de Regressão (Quantidade de Atropelamentos Agregados por Ano/Mês/UC)', fontsize=13)
    plt.xlabel('Número de Atropelamentos Registados')
    plt.ylabel('Frequência')
    plt.tight_layout()
    plt.show()

---
### 6. Estatísticas Descritivas das Variáveis Numéricas
Apresentação de medidas de tendência central (média, mediana), dispersão (desvio padrão, amplitude, intervalo interquartil) e forma (assimetria e curtose).

In [ ]:
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()

if colunas_numericas:
    estatisticas = df[colunas_numericas].describe().T
    estatisticas['mediana'] = df[colunas_numericas].median()
    estatisticas['IQR'] = estatisticas['75%'] - estatisticas['25%']
    estatisticas['assimetria (skew)'] = df[colunas_numericas].skew()
    estatisticas['curtose (kurt)'] = df[colunas_numericas].kurt()
    estatisticas['coef_variacao (%)'] = (estatisticas['std'] / estatisticas['mean'].replace(0, np.nan)) * 100
    
    colunas_ordem = ['count', 'mean', 'mediana', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR', 'assimetria (skew)', 'curtose (kurt)', 'coef_variacao (%)']
    display(estatisticas[colunas_ordem].round(2))
else:
    print("Nenhuma coluna numérica identificada.")

---
### 7. Distribuição Espacial dos Atropelamentos e Identificação de *Hotspots*
Utilizando as coordenadas geográficas UTM (`x` e `y`), analisamos a dispersão espacial das ocorrências, colorindo por Unidade de Conservação (`uc`) ou `classe`, além de um mapa de densidade (KDE / Hexbin) para evidenciar os *hotspots* de atropelamento.

In [ ]:
if {'x', 'y'}.issubset(df.columns):
    # Filtrar eventuais coordenadas nulas ou iguais a zero para visualização limpa
    df_espacial = df[(df['x'].notnull()) & (df['y'].notnull()) & (df['x'] > 0) & (df['y'] > 0)].copy()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Gráfico 1: Dispersão espacial por Classe ou UC
    hue_col = 'classe' if 'classe' in df_espacial.columns else None
    sns.scatterplot(
        data=df_espacial, x='x', y='y', hue=hue_col, 
        alpha=0.75, s=50, palette='tab10', ax=axes[0]
    )
    axes[0].set_title('Distribuição Espacial dos Atropelamentos (Coordenadas UTM)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Coordenada X (UTM Leste)')
    axes[0].set_ylabel('Coordenada Y (UTM Norte)')
    if hue_col:
        axes[0].legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left')
    
    # Gráfico 2: Mapa de Densidade (Hotspots) via Hexbin / KDE
    hb = axes[1].hexbin(df_espacial['x'], df_espacial['y'], gridsize=25, cmap='YlOrRd', mincnt=1)
    cb = fig.colorbar(hb, ax=axes[1])
    cb.set_label('Contagem de Atropelamentos (Densidade)')
    axes[1].set_title('Mapa de Densidade de Atropelamentos (Identificação de Hotspots)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Coordenada X (UTM Leste)')
    axes[1].set_ylabel('Coordenada Y (UTM Norte)')
    
    plt.tight_layout()
    plt.show()
    
    # Contagem por Unidade de Conservação (UC)
    if 'uc' in df.columns:
        plt.figure(figsize=(10, 4))
        ordem_uc = df['uc'].value_counts().index
        sns.countplot(data=df, y='uc', order=ordem_uc, palette='magma')
        plt.title('Total de Atropelamentos por Unidade de Conservação (UC)', fontsize=13, fontweight='bold')
        plt.xlabel('Quantidade de Registos')
        plt.ylabel('Unidade de Conservação (UC)')
        plt.tight_layout()
        plt.show()

---
### 8. Distribuição Temporal dos Atropelamentos
Análise da evolução dos atropelamentos ao longo dos anos (`ano`), sazonalidade mensal (`mes`), dias da semana (`dia_da_sem`), períodos (`periodo`) e horários (`hora`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Evolução por Ano
if 'ano' in df.columns:
    contagem_ano = df['ano'].value_counts().sort_index()
    sns.lineplot(x=contagem_ano.index.astype(str), y=contagem_ano.values, marker='o', linewidth=2.5, color='#2980b9', ax=axes[0, 0])
    axes[0, 0].set_title('Evolução Anual dos Atropelamentos', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Ano')
    axes[0, 0].set_ylabel('Quantidade de Atropelamentos')

# 2. Sazonalidade por Mês
if 'mes' in df.columns:
    ordem_meses = ['janeiro', 'fevereiro', 'marco', 'março', 'abril', 'maio', 'junho', 
                   'julho', 'agosto', 'setembro', 'outubro', 'novembro', 'dezembro']
    meses_presentes = [m for m in ordem_meses if m in df['mes'].astype(str).str.lower().unique()]
    if not meses_presentes:
        meses_presentes = df['mes'].value_counts().index.tolist()
    
    df_mes = df['mes'].astype(str).str.lower()
    sns.countplot(y=df_mes, order=meses_presentes, palette='viridis', ax=axes[0, 1])
    axes[0, 1].set_title('Distribuição Mensal dos Atropelamentos', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Quantidade')
    axes[0, 1].set_ylabel('Mês')

# 3. Distribuição por Dia da Semana
if 'dia_da_sem' in df.columns:
    ordem_dias = ['seg', 'ter', 'qua', 'qui', 'sex', 'sab', 'sáb', 'dom']
    dias_presentes = [d for d in ordem_dias if d in df['dia_da_sem'].astype(str).str.lower().unique()]
    if not dias_presentes:
        dias_presentes = df['dia_da_sem'].value_counts().index.tolist()
        
    df_dia = df['dia_da_sem'].astype(str).str.lower()
    sns.countplot(x=df_dia, order=dias_presentes, palette='crest', ax=axes[1, 0])
    axes[1, 0].set_title('Distribuição por Dia da Semana', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Dia da Semana')
    axes[1, 0].set_ylabel('Quantidade')

# 4. Distribuição por Período / Hora
col_tempo = 'hora' if 'hora' in df.columns else ('periodo' if 'periodo' in df.columns else None)
if col_tempo:
    sns.countplot(x=df[col_tempo].astype(str), palette='flare', ax=axes[1, 1], order=df[col_tempo].astype(str).value_counts().index)
    axes[1, 1].set_title(f'Distribuição por "{col_tempo}"', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel(col_tempo.capitalize())
    axes[1, 1].set_ylabel('Quantidade')
    axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## 9. Investigação de Possíveis Problemas nos Dados

Nesta secção investigamos sistematicamente os 7 problemas solicitados na especificação da Etapa 1:
1. **Classes desbalanceadas**
2. **Valores ausentes**
3. **Atributos com grande quantidade de valores `nao_identificado`**
4. **Outliers**
5. **Atributos redundantes**
6. **Correlação elevada entre atributos**
7. **Classes com quantidade insuficiente de exemplos**

### 9.1. Classes Desbalanceadas e Classes com Quantidade Insuficiente de Exemplos
Avaliamos a razão de desbalanceamento (*Imbalance Ratio*) nos atributos taxonómicos (`classe`, `ordem`, `familia`, `especie_nm`) e identificamos categorias raras (ex.: menos de 5 ou 10 exemplos), que inviabilizam a divisão estratificada em treino/validação/teste sem agrupamento prévio.

In [ ]:
atributos_taxonomicos = [c for c in ['silvestre_', 'classe', 'ordem', 'familia', 'especie_nm'] if c in df.columns]
LIMIAR_INSUFICIENTE = 5  # Classes com menos de 5 exemplos

resumo_desbalanceamento = []
for col in atributos_taxonomicos:
    contagens = df[col].dropna().value_counts()
    if len(contagens) > 1:
        maior_classe = contagens.iloc[0]
        menor_classe = contagens.iloc[-1]
        razao_desb = maior_classe / max(menor_classe, 1)
        qtd_insuficientes = (contagens < LIMIAR_INSUFICIENTE).sum()
        perc_insuficientes = (qtd_insuficientes / len(contagens)) * 100
        
        resumo_desbalanceamento.append({
            'Atributo': col,
            'Total de Categorias': len(contagens),
            'Classe Majoritária (Qtd)': f"{contagens.index[0]} ({maior_classe})",
            'Classe Minoritária (Qtd)': f"{contagens.index[-1]} ({menor_classe})",
            'Razão de Desbalanceamento (Max/Min)': round(razao_desb, 2),
            f'Categorias com < {LIMIAR_INSUFICIENTE} exemplos': qtd_insuficientes,
            f'% Categorias Insuficientes': round(perc_insuficientes, 2)
        })

df_desb = pd.DataFrame(resumo_desbalanceamento)
display(df_desb)

# Visualização das classes com poucos exemplos em 'ordem' e 'familia'
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col in zip(axes, [c for c in ['ordem', 'familia'] if c in df.columns][:2]):
    contagens = df[col].value_counts()
    cores = ['#e74c3c' if v < LIMIAR_INSUFICIENTE else '#3498db' for v in contagens.values]
    sns.barplot(x=contagens.values[:20], y=contagens.index[:20].astype(str), palette=cores[:20], ax=ax)
    ax.axvline(LIMIAR_INSUFICIENTE, color='red', linestyle='--', label=f'Mínimo recomendado ({LIMIAR_INSUFICIENTE} exemplos)')
    ax.set_title(f'Top 20 Categorias em "{col}" (Vermelho = Insuficiente)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Quantidade de Exemplos')
    ax.legend()

plt.tight_layout()
plt.show()

### 9.2. Valores Ausentes e Atributos com Grande Quantidade de `nao_identificado`
Filtramos diretamente os atributos que ultrapassam o patamar de atenção (> 15% de valores ausentes ou `nao_identificado`), os quais exigirão estratégias específicas no pré-processamento (como imputação, criação de categoria explícita ou descarte da coluna).

In [ ]:
atributos_criticos_ni = df_ausentes[df_ausentes['% Total Desconhecido'] >= 10.0]
print("Atributos com proporção relevante (>= 10%) de valores ausentes ou 'nao_identificado':")
display(atributos_criticos_ni)

### 9.3. Investigação de *Outliers* (Valores Atípicos)
Utilizamos o método do Intervalo Interquartil ($IQR = Q_3 - Q_1$, considerando *outliers* valores fora de $[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$) e *Boxplots* para inspecionar as variáveis numéricas (especialmente as coordenadas espaciais `x` e `y`, onde erros de digitação de GPS são comuns).

In [ ]:
resumo_outliers = []
cols_num_analise = [c for c in colunas_numericas if c != 'id']

for col in cols_num_analise:
    serie = df[col].dropna()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    
    outliers = serie[(serie < lim_inf) | (serie > lim_sup)]
    resumo_outliers.append({
        'Atributo': col,
        'Limite Inferior (IQR)': round(lim_inf, 2),
        'Limite Superior (IQR)': round(lim_sup, 2),
        'Qtd Outliers': len(outliers),
        '% Outliers': round((len(outliers) / len(serie)) * 100, 2)
    })

display(pd.DataFrame(resumo_outliers))

# Boxplots das variáveis numéricas
if cols_num_analise:
    fig, axes = plt.subplots(1, len(cols_num_analise), figsize=(4 * len(cols_num_analise), 5))
    if len(cols_num_analise) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols_num_analise):
        sns.boxplot(y=df[col], ax=ax, color='#9b59b6', width=0.4)
        ax.set_title(f'Boxplot: {col}', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 9.4. Atributos Redundantes e Correlação Elevada entre Atributos
Analisamos:
1. **Correlação Linear (Pearson) e Monotónica (Spearman)** entre variáveis numéricas.
2. **Associação entre Variáveis Categóricas (V de Cramér)** para detetar redundâncias estruturais na base de dados, tais como:
   * `id` (identificador único sem valor preditivo);
   * `especie_nm` (nome científico) vs. `nome_popul` (nome popular) — mapeamento 1-para-1 redundante;
   * Hierarquia taxonómica determinística (`especie_nm` $\rightarrow$ `familia` $\rightarrow$ `ordem` $\rightarrow$ `classe`);
   * `data_dt` vs. (`ano`, `mes`, `dia_da_sem`, `periodo`).

In [ ]:
from scipy.stats import chi2_contingency

# 1. Matriz de Correlação Numérica (Pearson e Spearman)
if len(cols_num_analise) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    corr_pearson = df[cols_num_analise].corr(method='pearson')
    sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0])
    axes[0].set_title('Correlação Numérica (Pearson)', fontsize=12, fontweight='bold')
    
    corr_spearman = df[cols_num_analise].corr(method='spearman')
    sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1])
    axes[1].set_title('Correlação Numérica (Spearman)', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# 2. Matriz de Associação Categórica (V de Cramér) para detetar atributos redundantes
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] < 2 or confusion_matrix.shape[1] < 2:
        return 0.0
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    denom = min((kcorr-1), (rcorr-1))
    return np.sqrt(phi2corr / denom) if denom > 0 else 0.0

cols_cat = [c for c in df.columns if c not in ['id', 'x', 'y', 'data_dt']]
matriz_cramer = pd.DataFrame(index=cols_cat, columns=cols_cat, dtype=float)

for c1 in cols_cat:
    for c2 in cols_cat:
        if c1 == c2:
            matriz_cramer.loc[c1, c2] = 1.0
        else:
            matriz_cramer.loc[c1, c2] = cramers_v(df[c1].astype(str), df[c2].astype(str))

plt.figure(figsize=(12, 9))
sns.heatmap(matriz_cramer, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1)
plt.title('Matriz de Associação V de Cramér (Deteção de Redundância e Correlação Elevada)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Listar pares de atributos com associação/correlação muito elevada (>= 0.75)
pares_redundantes = []
for i in range(len(cols_cat)):
    for j in range(i+1, len(cols_cat)):
        val = matriz_cramer.iloc[i, j]
        if val >= 0.75:
            pares_redundantes.append({
                'Atributo 1': cols_cat[i],
                'Atributo 2': cols_cat[j],
                'V de Cramér': round(val, 3)
            })

if pares_redundantes:
    print("Pares de atributos com forte associação / potencial redundância (V de Cramér >= 0.75):")
    display(pd.DataFrame(pares_redundantes).sort_values(by='V de Cramér', ascending=False))

---
### 10. Síntese das Conclusões da Etapa 1 (Diagnóstico para o Pré-processamento)

1. **Atributos Irrelevantes / Redundantes:**
   * O atributo `id` é apenas um identificador sequencial e deve ser removido antes da modelagem.
   * `especie_nm` e `nome_popul` apresentam correspondência quase perfeita (redundância semântica).
   * Existe dependência hierárquica estrita entre `especie_nm` $\rightarrow$ `familia` $\rightarrow$ `ordem` $\rightarrow$ `classe`; caso `classe` seja a variável alvo, `ordem`, `familia`, `especie_nm` e `nome_popul` **não podem ser usados como variáveis preditoras** para evitar *Data Leakage* (vazamento de dados).
   * `data_dt` já está decomposta em `ano`, `mes` e `dia_da_sem`, podendo ser convertida em atributos cíclicos (seno/cosseno do dia do ano) ou descartada após extração.
2. **Valores `nao_identificado` e Ausentes:**
   * Atributos como `hora`, `sexo`, `posicao` e níveis taxonómicos mais baixos (`especie_nm`) costumam concentrar elevada proporção de `nao_identificado` devido ao estado de conservação das carcaças encontradas nas rodovias.
3. **Desbalanceamento e Classes Raras:**
   * Enquanto `classe` possui poucas categorias bem representadas (embora desbalanceadas a favor de aves/mamíferos), atributos como `familia` e `especie_nm` possuem dezenas de classes com quantidade insuficiente de exemplos ($< 5$), exigindo agrupamento na categoria `'Outros'` caso sejam utilizados.
4. **Outliers Espaciais:**
   * As coordenadas UTM (`x` e `y`) permitem mapear claramente os *hotspots* de atropelamento próximos às Unidades de Conservação, mas eventuais pontos fora dos limites geográficos do Distrito Federal devem ser tratados na etapa de limpeza.